In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
import random

# Configurar random seed para reprodutibilidade
np.random.seed(42)

def criar_dataset_realista(n_amostras=500):
    provincias = ['Luanda', 'Benguela', 'Huíla', 'Huambo', 'Cabinda', 'Cunene', 'Moxico', 'Uíge']

    dados = []

    for i in range(n_amostras):
        provincia = random.choice(provincias)

        # Coordenadas realistas por província
        coords = {
            'Luanda': (-8.839, 13.289),
            'Benguela': (-12.576, 13.406),
            'Huíla': (-14.917, 13.933),
            'Huambo': (-12.776, 15.739),
            'Cabinda': (-5.550, 12.197),
            'Cunene': (-16.317, 15.800),
            'Moxico': (-11.850, 20.000),
            'Uíge': (-7.617, 15.050)
        }

        lat, lon = coords[provincia]

        # Adicionar variação nas coordenadas
        lat += np.random.normal(0, 0.1)
        lon += np.random.normal(0, 0.1)

        # Variáveis meteorológicas com relações mais complexas
        mes = random.randint(1, 12)
        ano = random.choice([2023, 2024, 2025])

        # CORREÇÃO: Padrão sazonal correto para Angola
        # Estação chuvosa: Outubro a Abril | Estação seca: Maio a Setembro
        if mes in [10, 11, 12, 1, 2, 3, 4]:  # Estação chuvosa corrigida
            precipitacao_base = np.random.normal(150, 50)
            temp_base = np.random.normal(26, 2)
        else:  # Estação seca (Maio a Setembro)
            precipitacao_base = np.random.normal(50, 20)
            temp_base = np.random.normal(28, 2)

        # Ajustar por província considerando variações regionais
        ajuste_provincia = {
            'Luanda': {'precip': -20, 'temp': +2},      # Litoral - mais ameno
            'Benguela': {'precip': +10, 'temp': +1},    # Litoral - mais chuvas
            'Huíla': {'precip': +30, 'temp': -2},       # Interior - mais ameno
            'Huambo': {'precip': +40, 'temp': -1},      # Altitude - clima suave
            'Cabinda': {'precip': +50, 'temp': +1},     # Litoral norte - mais chuvas
            'Cunene': {'precip': -10, 'temp': +3},      # Sudoeste - semiárido
            'Moxico': {'precip': +20, 'temp': 0},       # Interior
            'Uíge': {'precip': +60, 'temp': 0}          # Norte - mais chuvas
        }

        precip_ajuste = ajuste_provincia[provincia]['precip']
        temp_ajuste = ajuste_provincia[provincia]['temp']

        precipitacao = max(0, precipitacao_base + precip_ajuste + np.random.normal(0, 30))
        temperatura = temp_base + temp_ajuste + np.random.normal(0, 1.5)
        humidade = max(30, min(95, 60 + (precipitacao/5) + np.random.normal(0, 10)))
        vento = max(5, np.random.normal(12, 4))
        rad_solar = max(100, 250 - (precipitacao/2) + np.random.normal(0, 25))

        # Critério de risco de inundação ajustado
        risco_precip = 1 if precipitacao > 180 else 0
        risco_humidade = 1 if humidade > 85 else 0
        # CORREÇÃO: Período de maior risco durante a estação chuvosa
        risco_tempo = 1 if mes in [10, 11, 12, 1, 2, 3, 4] else 0
        risco_provincia = 1 if provincia in ['Cabinda', 'Uíge', 'Huambo'] else 0

        total_risco = risco_precip + risco_humidade + risco_tempo + risco_provincia

        # Probabilidade de flood baseada em múltiplos fatores
        prob_flood = min(0.95, total_risco * 0.2 + (precipitacao/400) + (humidade/100)*0.3)
        flood = 1 if random.random() < prob_flood else 0

        # Adicionar algum ruído (casos não explicáveis)
        if random.random() < 0.05:  # 5% de ruído
            flood = 1 - flood

        dados.append([
            provincia, lat, lon, ano, mes, precipitacao, temperatura,
            humidade, vento, rad_solar, flood
        ])

    df = pd.DataFrame(dados, columns=[
        'provincia', 'latitude', 'longitude', 'ano', 'mes',
        'precipitacao_mm', 'temperatura_C', 'humidade_percent',
        'vento_kmh', 'rad_solar_Wm2', 'flood'
    ])

    return df

# Criar dataset corrigido
df_corrigido = criar_dataset_realista(500)

# Salvar dataset corrigido
df_corrigido.to_csv('dataset_angola_corrigido.csv', index=False)

print("Dataset corrigido criado com 500 amostras")
print(f"Distribuição do target: {df_corrigido['flood'].value_counts().to_dict()}")
print("Estações corretamente ajustadas para o clima de Angola:")
print("- Estação chuvosa: Outubro a Abril")
print("- Estação seca (Cacimbo): Maio a Setembro")

Dataset corrigido criado com 500 amostras
Distribuição do target: {1: 331, 0: 169}
Estações corretamente ajustadas para o clima de Angola:
- Estação chuvosa: Outubro a Abril
- Estação seca (Cacimbo): Maio a Setembro
